# Comparison of deconvolver calibrators

In this notebook, we compare the performance of the our linear calibrator
versus temperature scaling, vector scaling and dirichlet calibration.

## Imports and utility functions

In [3]:
import os
import sys
from pathlib import Path
import json
from functools import reduce

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.model_selection import KFold, ParameterGrid
from scipy.stats import linregress

sys.path.append('..')

from EDA.edautils import plot_deconvolution_results
from syto.deconvolution.evaluation import compute_deconvolution_metrics
from syto.calibration.linear_calibrator import LinearCalibrator
from EDA.dirichlet_calibration import DirichletCalibrator

%load_ext autoreload
%autoreload 2

In [4]:
def prod_len_dict_keys(dict_: dict) -> int:
    return reduce(lambda x, y: x * y, list(map(len, dict_.values())), 1)

In [5]:
def plot_mixtures_pred_vs_true(
    ground_truth_mixture: np.ndarray,
    predicted_mixtures: list[np.ndarray],
    predicted_mixture_labels: list[str],
    width: float = 0.2,
    title: str = "Predicted Mixtures vs Ground Truth Mixture",
    ctype_names: list[str] = None,
):
    """
    Plot the predicted mixtures against the ground truth mixture for a single sample.
    The function creates a bar plot where the x axis represents the cell types and the y axis represents the mixture proportions.
    Each predicted mixture is plotted as a separate bar. The ground truth mixture is plotted as red dots for each cell type.

    Args:
        ground_truth_mixture: A 1D array of shape (n_cell_types,) representing the true mixture proportions.
        predicted_mixtures: A list of 1D arrays, each of shape (n_cell_types,), representing the predicted mixture proportions from different models.
        predicted_mixture_labels: A list of strings representing the labels for each predicted mixture (e.g., model names).
    """
    assert len(predicted_mixtures) == len(
        predicted_mixture_labels
    ), "Number of predicted mixtures must match number of labels"
    assert all(
        pred.shape == ground_truth_mixture.shape for pred in predicted_mixtures
    ), "All predicted mixtures must have the same shape as the ground truth mixture"
    plt.figure(figsize=(8, 6))
    n_cell_types = len(ground_truth_mixture)
    n_predicted_mixtures = len(predicted_mixtures)
    x = np.arange(n_cell_types)  # start of the the label locations
    x_center = (
        x + width * (n_predicted_mixtures - 1) / 2
    )  # middle of the group of bars for each cell type

    # Plot the ground truth mixture as red dots
    plt.scatter(
        x_center, ground_truth_mixture, color="red", zorder=-1, label="Ground Truth"
    )

    # Plot each predicted mixture as a bar
    for i, (predicted_mixture, label) in enumerate(
        zip(predicted_mixtures, predicted_mixture_labels)
    ):
        plt.bar(x + i * width, predicted_mixture, width, label=label, alpha=0.7)

    # plot the cell types names on the x axis, rotated by 90 degrees
    if ctype_names is not None:
        plt.xticks(x_center, ctype_names, rotation=90)
    plt.ylabel("Mixture Proportions")
    plt.title(title)
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.show()

In [6]:
def plot_heatmap(
    matrix: np.ndarray,
    title: str = "Heatmap",
    color_bar_label: str = "Probability",
    xlabel="Predicted Class",
    ylabel="True Class",
    x_ticks: list[str] = None,
    y_ticks: list[str] = None,
    vmin: float | None = None,
    vmax: float | None = None,
):
    """Plot a heatmap of the given matrix with cell type names on the axes."""
    plt.figure(figsize=(10, 8))
    plt.imshow(matrix, cmap="viridis", aspect="auto", vmin=vmin, vmax=vmax)
    plt.colorbar(label=color_bar_label)
    plt.xticks(
        ticks=np.arange(matrix.shape[1]),
        labels=(
            x_ticks if x_ticks is not None else [str(i) for i in range(matrix.shape[1])]
        ),
        rotation=90,
    )
    plt.yticks(
        ticks=np.arange(matrix.shape[0]),
        labels=(
            y_ticks if y_ticks is not None else [str(i) for i in range(matrix.shape[0])]
        ),
    )
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

## Data loading

In [7]:
with open("../App/labels_dict.json", "r", encoding="utf-8") as f:
    labels_to_ctype_names = json.load(f)
labels_to_ctype_names = {int(k): v for k, v in labels_to_ctype_names.items()}
ctype_names_to_labels = {v: k for k, v in labels_to_ctype_names.items()}
ctype_names_list = list(ctype_names_to_labels.keys())

In [8]:
DATA_DIR = Path("/home/nathan/Documents/Scolaire/7_Cesure/2_KU_Leuven/syto/Data/uxm_val_test_data")

In [9]:
uxm_uncal_pred_and_target = np.load(DATA_DIR / "uncalibrated_predictions.npz")
uxm_uncal_val_data = uxm_uncal_pred_and_target["val_pred"]
uxm_uncal_test_data = uxm_uncal_pred_and_target["test_pred"]
uxm_val_targets = uxm_uncal_pred_and_target["val_target"]
uxm_test_targets = uxm_uncal_pred_and_target["test_target"]

In [10]:
def wrapper_comp_metrics(
    test_pred: dict, test_targets: np.ndarray, round_: int = 6
) -> pd.DataFrame:
    results = {
        method_name: compute_deconvolution_metrics(
            pred=pred, target=test_targets, class_names=ctype_names_list
        )
        for method_name, pred in test_pred.items()
    }
    results = {
        k: {metric: value for metric, value in v.items() if "per_class" not in metric}
        for k, v in results.items()
    }
    results_df = pd.DataFrame(results).T
    float_cols = [
        "mae",
        "mse",
        "kl",
        "max_error",
        "cosine_sim",
        "loa_lower",
        "loa_upper",
        "loa_width",
        "worst_class_loa_lower",
        "worst_class_loa_upper",
        "worst_class_loa_width",
    ]
    for col in float_cols:
        results_df[col] = results_df[col].astype(float).round(round_)
    results_df.sort_values("mse", inplace=True)
    return results_df, results

## Compare calibrators

### Trained calibrators

In [11]:
# global settings
SCHEDULER = "plateau"
PLATEAU_FACTOR = 0.5
PLATEAU_PATIENCE = 10
LOG_TRANSFORM = True
BATCH_SIZE = None  # whole dataset
OPTIMIZER = "adam"
PATIENCE = 15
TOL = 1e-4
N_FOLDS = 3
DEVICE = "cuda"
CALIBRATOR_PARAMS = {
    "scheduler": SCHEDULER,
    "log_transform": LOG_TRANSFORM,
    "batch_size": BATCH_SIZE,
    "optimizer": OPTIMIZER,
    "patience": PATIENCE,
    "plateau_factor": PLATEAU_FACTOR,
    "plateau_patience": PLATEAU_PATIENCE,
    "tol": TOL,
    "verbose": False,
    "device": DEVICE,
}

### Trained calibrators

#### Temperature scaling

In [12]:
# Temperature scaling on the log prob grid search
METHOD = "temperature"
param_grid = ParameterGrid(
    {
        "reg_lambda": [0.0, 1e-4],
        "reg_mu": [None],
        "lr": [1e-1, 1e-2, 1e-3, 1e-4],
        "max_iter": [500],
    }
)

results = []
n_loops = len(param_grid) * N_FOLDS
best_val_loss = np.inf
best_temp_scal_params = None
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
with tqdm(total=n_loops) as pbar:
    for params in param_grid:
        lr = params["lr"]
        max_iter = params["max_iter"]
        reg_lambda = params["reg_lambda"]
        reg_mu = params["reg_mu"]
        fold_val_losses = []
        fold_val_mses = []
        max_n_trained_epochs = 0
        for train_index, val_index in kf.split(uxm_uncal_val_data):
            X_tr_fold, X_val_fold = (
                uxm_uncal_val_data[train_index],
                uxm_uncal_val_data[val_index],
            )
            y_tr_fold, y_val_fold = (
                uxm_val_targets[train_index],
                uxm_val_targets[val_index],
            )
            temp_scal = DirichletCalibrator(
                method=METHOD,
                lr=lr,
                max_iter=max_iter,
                reg_lambda=reg_lambda,
                reg_mu=reg_mu,
                **CALIBRATOR_PARAMS
            )
            temp_scal.fit(
                X=X_tr_fold,
                y=y_tr_fold,
                X_val=X_val_fold,
                y_val=y_val_fold,
                report_every=10,
            )
            fold_val_losses.append(temp_scal.best_metrics_["val_loss"])
            fold_val_mses.append(temp_scal.best_metrics_["val_mse"])
            max_n_trained_epochs = max(
                max_n_trained_epochs, temp_scal.history_["epoch"][-1] + 1
            )
            pbar.update(1)
        avg_best_val_loss = np.mean(fold_val_losses)
        results.append(
            {
                # Hyperparameters
                "reg_lambda": reg_lambda,
                "reg_mu": reg_mu,
                "lr": lr,
                "max_iter": max_iter,
                # Key metrics
                "avg_best_val_loss": avg_best_val_loss,
                "avg_best_val_mse": np.mean(fold_val_mses),
                "max_n_epochs_trained": max_n_trained_epochs,
                # Fitted model (not in DataFrame columns, but accessible)
            }
        )
        if avg_best_val_loss < best_val_loss:
            best_val_loss = avg_best_val_loss
            best_temp_scal_params = params
results_df_temp_scal = pd.DataFrame(results)
results_df_temp_scal.sort_values("avg_best_val_loss", inplace=True)
results_df_temp_scal

  0%|          | 0/24 [00:00<?, ?it/s]

,reg_lambda,reg_mu,lr,max_iter,avg_best_val_loss,avg_best_val_mse,max_n_epochs_trained
4,0.0000,None,0.0010,500,1.488492,0.000264,262
5,0.0001,None,0.0010,500,1.488492,0.000264,262
2,0.0000,None,0.0100,500,1.488522,0.000264,36
3,0.0001,None,0.0100,500,1.488522,0.000264,36
0,0.0000,None,0.1000,500,1.488536,0.000264,17
1,0.0001,None,0.1000,500,1.488536,0.000264,17
6,0.0000,None,0.0001,500,1.516646,0.000343,500
7,0.0001,None,0.0001,500,1.516646,0.000343,500


In [13]:
best_temp_scal_params

{'lr': 0.001, 'max_iter': 500, 'reg_lambda': 0.0, 'reg_mu': None}

#### Vector scaling on the log prop

In [14]:
# Vector scaling on the log prob grid search
METHOD = "diagonal"
param_grid = ParameterGrid(
    {
        "reg_lambda": [0.0, 1e-3, 1e-2, 1e-1],
        "reg_mu": [None],
        "lr": [1e-2, 1e-4, 1e-3],
        "max_iter": [1000],
    }
)

results = []
n_loops = len(param_grid) * N_FOLDS
best_val_loss = float("inf")
best_vect_scal_params = None
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
with tqdm(total=n_loops) as pbar:
    for params in param_grid:
        lr = params["lr"]
        max_iter = params["max_iter"]
        reg_lambda = params["reg_lambda"]
        reg_mu = params["reg_mu"]
        fold_val_losses = []
        fold_val_mses = []
        max_n_trained_epochs = 0
        for train_index, val_index in kf.split(uxm_uncal_val_data):
            X_tr_fold, X_val_fold = (
                uxm_uncal_val_data[train_index],
                uxm_uncal_val_data[val_index],
            )
            y_tr_fold, y_val_fold = (
                uxm_val_targets[train_index],
                uxm_val_targets[val_index],
            )
            vect_log_dir_cal = DirichletCalibrator(
                method=METHOD,
                lr=lr,
                max_iter=max_iter,
                reg_lambda=reg_lambda,
                reg_mu=reg_mu,
                **CALIBRATOR_PARAMS
            )
            vect_log_dir_cal.fit(
                X=X_tr_fold,
                y=y_tr_fold,
                X_val=X_val_fold,
                y_val=y_val_fold,
                report_every=10,
            )
            fold_val_losses.append(vect_log_dir_cal.best_metrics_["val_loss"])
            fold_val_mses.append(vect_log_dir_cal.best_metrics_["val_mse"])
            max_n_trained_epochs = max(
                max_n_trained_epochs, vect_log_dir_cal.history_["epoch"][-1] + 1
            )
            pbar.update(1)
        avg_val_loss = np.mean(fold_val_losses)
        results.append(
            {
                # Hyperparameters
                "reg_lambda": reg_lambda,
                "reg_mu": reg_mu,
                "lr": lr,
                "max_iter": max_iter,
                # Key metrics
                "avg_best_val_loss": avg_val_loss,
                "avg_best_val_mse": np.mean(fold_val_mses),
                "max_n_epochs_trained": max_n_trained_epochs,
            }
        )
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_vect_scal_params = params

results_df_vect_log_dir_cal = pd.DataFrame(results)
results_df_vect_log_dir_cal.sort_values("avg_best_val_loss", inplace=True)
results_df_vect_log_dir_cal

  0%|          | 0/36 [00:00<?, ?it/s]

,reg_lambda,reg_mu,lr,max_iter,avg_best_val_loss,avg_best_val_mse,max_n_epochs_trained
9,0.001,None,0.0010,1000,1.484678,0.000236,260
8,0.000,None,0.0010,1000,1.485511,0.000240,239
2,0.010,None,0.0100,1000,1.487042,0.000249,37
10,0.010,None,0.0010,1000,1.487067,0.000249,288
1,0.001,None,0.0100,1000,1.488272,0.000242,37
0,0.000,None,0.0100,1000,1.489450,0.000247,35
4,0.000,None,0.0001,1000,1.497126,0.000286,1000
5,0.001,None,0.0001,1000,1.497248,0.000287,1000
6,0.010,None,0.0001,1000,1.499592,0.000293,1000
3,0.100,None,0.0100,1000,1.510419,0.000318,30


In [15]:
best_vect_scal_params

{'lr': 0.001, 'max_iter': 1000, 'reg_lambda': 0.001, 'reg_mu': None}

#### MS on the log prob

In [16]:
# Matrix scaling on the log prob (Dir cal) grid search
METHOD = "full"
param_grid = ParameterGrid(
    {
        "reg_lambda": [1000, 5000, 10000],
        "reg_mu": [None, 0, 1],
        "lr": [1e-4, 1e-5, 1e-6],
        "max_iter": [500],
    }
)

results = []
n_loops = len(param_grid) * N_FOLDS
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
best_val_loss = float("inf")
best_dir_cal_params = None
with tqdm(total=n_loops) as pbar:
    for params in param_grid:
        lr = params["lr"]
        max_iter = params["max_iter"]
        reg_lambda = params["reg_lambda"]
        reg_mu = params["reg_mu"]
        fold_val_losses = []
        fold_val_mses = []
        max_n_trained_epochs = 0
        for train_index, val_index in kf.split(uxm_uncal_val_data):
            X_tr_fold, X_val_fold = (
                uxm_uncal_val_data[train_index],
                uxm_uncal_val_data[val_index],
            )
            y_tr_fold, y_val_fold = (
                uxm_val_targets[train_index],
                uxm_val_targets[val_index],
            )
            mat_log_dir_cal = DirichletCalibrator(
                method=METHOD,
                lr=lr,
                max_iter=max_iter,
                reg_lambda=reg_lambda,
                reg_mu=reg_mu,
                **CALIBRATOR_PARAMS
            )
            mat_log_dir_cal.fit(
                X=X_tr_fold,
                y=y_tr_fold,
                X_val=X_val_fold,
                y_val=y_val_fold,
                report_every=10,
            )
            fold_val_losses.append(mat_log_dir_cal.best_metrics_["val_loss"])
            fold_val_mses.append(mat_log_dir_cal.best_metrics_["val_mse"])
            max_n_trained_epochs = max(
                max_n_trained_epochs, mat_log_dir_cal.history_["epoch"][-1] + 1
            )
            pbar.update(1)
        avg_val_loss = np.mean(fold_val_losses)
        results.append(
            {
                # Hyperparameters
                "reg_lambda": reg_lambda,
                "reg_mu": reg_mu,
                "lr": lr,
                "max_iter": max_iter,
                # Key metrics
                "avg_best_val_loss": avg_val_loss,
                "avg_best_val_mse": np.mean(fold_val_mses),
                "max_n_epochs_trained": max_n_trained_epochs,
            }
        )
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_dir_cal_params = params
results_df_mat_log_dir_cal = pd.DataFrame(results)
results_df_mat_log_dir_cal.sort_values("avg_best_val_loss", inplace=True)
results_df_mat_log_dir_cal

  0%|          | 0/81 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
best_dir_cal_params

### Compare metrics across calibrators

In [ ]:
best_temp_scal_params = {"lr": 1e-03, "max_iter": 500, "reg_lambda": 0, "reg_mu": None}
best_vect_scal_params = {
    "lr": 1e-03,
    "max_iter": 500,
    "reg_lambda": 1e-3,
    "reg_mu": None,
}
best_mat_dir_cal_params = {
    "lr": 1e-05,
    "max_iter": 500,
    "reg_lambda": 5000,
    "reg_mu": None,
}

We train with the best hyperparameters for each calibrator.

In [ ]:
# Temperature scaling
best_temp_scal = DirichletCalibrator(
    method="temperature", **best_temp_scal_params, **CALIBRATOR_PARAMS
)
best_temp_scal.fit(
    X=uxm_uncal_val_data, y=uxm_val_targets, report_every=10
)

In [ ]:
# Vector scaling
best_vect_scal = DirichletCalibrator(
    method="diagonal", **best_vect_scal_params, **CALIBRATOR_PARAMS
)
best_vect_scal.fit(
    X=uxm_uncal_val_data, y=uxm_val_targets, report_every=10
)

In [ ]:
# Dirichlet calibration
best_mat_dir_cal = DirichletCalibrator(
    method="full", **best_mat_dir_cal_params, **CALIBRATOR_PARAMS
)
best_mat_dir_cal.fit(
    X=uxm_uncal_val_data, y=uxm_val_targets, report_every=10
)

In [ ]:
# linear calibrators
linear_cal = LinearCalibrator()
linear_cal.fit(uxm_uncal_val_data, uxm_val_targets)

In [ ]:
# inferences with the best calibrators
uxm_test_pred_lin_cal_clip0_norm, _ = linear_cal.predict(
    uxm_uncal_test_data, norm_method="clip0-normalize"
)
uxm_test_pred_lin_cal_simplex_proj, _ = linear_cal.predict(
    uxm_uncal_test_data, norm_method="simplex-projection"
)
uxm_test_pred_temp_scal = best_temp_scal.predict(uxm_uncal_test_data)
uxm_test_pred_vect_scal = best_vect_scal.predict(uxm_uncal_test_data)
uxm_test_pred_mat_dir_cal = best_mat_dir_cal.predict(uxm_uncal_test_data)

In [ ]:
results_df, results_dict = wrapper_comp_metrics(
    {
        "No calibration": uxm_uncal_test_data,
        "Linear Cal. (clip0-norm)": uxm_test_pred_lin_cal_clip0_norm,
        "Linear Cal. (simplex-projection)": uxm_test_pred_lin_cal_simplex_proj,
        "Temperature scaling": uxm_test_pred_temp_scal,
        "Vector scaling": uxm_test_pred_vect_scal,
        "Dirichlet calibration": uxm_test_pred_mat_dir_cal,
    },
    test_targets=uxm_test_targets,
)
results_df.sort_values("mse")

In [ ]:
# print the latex formatting for the results dataframe
def fmt_r2(val):
    return f"{val * 100:.2f}"


def fmt_loa(lower, upper):
    return f"[{lower*1e2:.2f}, {upper*1e2:.2f}]"


def fmt_val(val, decimals=6, mult_factor=1):
    return f"{val * mult_factor:.{decimals}f}"


metrics_config = {
    "r2": {
        "col": "overall_r2",
        "higher_better": True,
        "fmt": lambda row: fmt_r2(row["overall_r2"]),
    },
    "loa": {
        "col": "loa_width",
        "higher_better": False,
        "fmt": lambda row: fmt_loa(row["loa_lower"], row["loa_upper"]),
    },
    "loa_worst": {
        "col": "worst_class_loa_width",
        "higher_better": False,
        "fmt": lambda row: fmt_loa(
            row["worst_class_loa_lower"], row["worst_class_loa_upper"]
        ),
    },
    "mae": {
        "col": "mae",
        "higher_better": False,
        "fmt": lambda row: fmt_val(row["mae"], mult_factor=1e3, decimals=2),
    },
    "mse": {
        "col": "mse",
        "higher_better": False,
        "fmt": lambda row: fmt_val(row["mse"], mult_factor=1e4, decimals=2),
    },
    "kl": {
        "col": "kl",
        "higher_better": False,
        "fmt": lambda row: fmt_val(row["kl"], decimals=2, mult_factor=1e2),
    },
}

# Find best and second best for each metric
rankings = {}
for key, config in metrics_config.items():
    col = config["col"]
    ascending = not config["higher_better"]
    sorted_idx = (
        results_df[col].astype(float).sort_values(ascending=ascending).index.tolist()
    )
    rankings[key] = {"best": sorted_idx[0], "second": sorted_idx[1]}

# Generate LaTeX content lines
for method_name, row in results_df.iterrows():
    values = []
    for key, config in metrics_config.items():
        formatted = config["fmt"](row)
        if method_name == rankings[key]["best"]:
            formatted = f"\\textbf{{{formatted}}}"
        elif method_name == rankings[key]["second"]:
            formatted = f"\\underline{{{formatted}}}"
        values.append(formatted)
    line = f"      {method_name:<40s} & {' & '.join(values)} \\\\"
    print(line)

## Avg MSE and KL per mixture complexity

In [ ]:
assert False

In [ ]:
## Computing MSE and KL divergence per mixture complexity (1 ctype to 10 ctypes)
model_name_to_preds = {
    "No calibration": uxm_uncal_test_data,
    "Linear Cal. (clip0-norm)": uxm_test_pred_lin_cal_clip0_norm,
    "Linear Cal. (simplex-projection)": uxm_test_pred_lin_cal_simplex_proj,
    "Temperature scaling": uxm_test_pred_temp_scal,
    "Vector scaling": uxm_test_pred_vect_scal,
    "Dirichlet calibration": uxm_test_pred_mat_dir_cal,
}
n_ctypes_in_mixture = []
mse_for_n_ctypes = {model_name: [] for model_name in model_name_to_preds.keys()}
kl_for_n_ctypes = {model_name: [] for model_name in model_name_to_preds.keys()}
for i in range(1, 10 + 1):
    mask = np.where(np.sum(uxm_test_targets > 0, axis=1) == i)[0]
    n_ctypes_in_mixture.append(i)
    for model_name, preds in model_name_to_preds.items():
        sub_pred = preds[mask]
        sub_target = uxm_test_targets[mask]
        deconv_metrics = compute_deconvolution_metrics(
            pred=sub_pred, target=sub_target, class_names=ctype_names_list
        )
        mse_for_n_ctypes[model_name].append(deconv_metrics["mse"])
        kl_for_n_ctypes[model_name].append(deconv_metrics["kl"])

In [ ]:
# plot MSE and KL divergence as a function of the number of cell types in the mixture, with one line per model
plt.figure(figsize=(20, 8))
plt.subplot(1, 2, 1)
color_per_model = {
    "No calibration": "blue",
    "Linear Cal. (clip0-norm)": "green",
    "Linear Cal. (simplex-projection)": "red",
    "Temperature scaling": "purple",
    "Vector scaling": "brown",
    "Dirichlet calibration": "pink",
}
marker_per_model = {
    "No calibration": "o",
    "Linear Cal. (clip0-norm)": "D",
    "Linear Cal. (simplex-projection)": "^",
    "Temperature scaling": "v",
    "Vector scaling": "<",
    "Dirichlet calibration": ">",
}
for model_name, mses in mse_for_n_ctypes.items():
    mean_mse = np.mean(mses)
    plt.axhline(
        y=mean_mse,
        color=color_per_model[model_name],
        linestyle="--",
        label=f"{model_name} (avg MSE: {mean_mse*1e4:.2f})",
    )
    plt.plot(
        n_ctypes_in_mixture,
        mses,
        marker=marker_per_model[model_name],
        color=color_per_model[model_name],
        label=model_name,
    )
plt.xlabel("Number of Cell Types in Mixture")
plt.ylabel("MSE")
plt.ylim(0, None)
plt.xticks(n_ctypes_in_mixture)
plt.title("MSE vs Number of Cell Types in Mixture")
plt.legend()
plt.grid()
plt.subplot(1, 2, 2)
for model_name, kls in kl_for_n_ctypes.items():
    mean_kl = np.mean(kls)
    plt.axhline(
        y=mean_kl,
        color=color_per_model[model_name],
        linestyle="--",
        label=f"{model_name} (avg KL: {mean_kl*1e1:.2f})",
    )
    plt.plot(
        n_ctypes_in_mixture,
        kls,
        marker=marker_per_model[model_name],
        color=color_per_model[model_name],
        label=model_name,
    )
plt.xlabel("Number of Cell Types in Mixture")
plt.ylabel("KL Divergence")
plt.ylim(0, None)
plt.xticks(n_ctypes_in_mixture)
plt.title("KL Divergence vs Number of Cell Types in Mixture")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
len(np.where(np.sum(target_prop > 0, axis=1) == 11)[0])